# NBA Predictive Model Training

This notebook trains and evaluates machine learning models for NBA team scoring prediction.

It combines engineered historical features, applies dimensionality reduction and regression techniques, and evaluates predictive performance using a season-based validation approach.

## Modeling Workflow

1. Feature preparation and preprocessing
2. Model training and dimensionality reduction
3. Historical validation and performance evaluation
4. Probability estimation and market analysis

## 1. Model Training and Historical Evaluation

Builds the predictive model using historical NBA data, evaluates its performance across seasons, and estimates game outcome probabilities in relation to sportsbook market lines.

In [ ]:
# ===============================
# FACTOR ARCHITECTURE v2 (FIXED)
# ===============================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge

from scipy.stats import t

# ===============================
# LOAD DATA
# ===============================

df = pd.read_parquet("training_dataset.parquet").copy()
df = df.sort_values("Date").reset_index(drop=True)

df["PTS_target"] = df["PTS_target"].clip(60, 160)
df = df[df["PTS_target"].notna()]
df = df[df["Market_TT"].notna()]
# ===============================
# FEATURE BLOCKS
# ===============================

ATTACK = [
    "Rolling_PTS_for_5", "Rolling_PTS_for_15",
    "Rolling_ORtg_5", "Rolling_ORtg_15",
    "Rolling_eFG%_5", "Rolling_eFG%_15",
    "Rolling_TS%_5", "Rolling_TS%_15",
    "Rolling_FT/FGA_5", "Rolling_FT/FGA_15",
    "Derived_Off_vs_Def_5", "Derived_Off_vs_Def_15"
]

DEFENSE = [
    "Rolling_PTS_against_5", "Rolling_PTS_against_15",
    "Rolling_DRtg_5", "Rolling_DRtg_15",
    "Rolling_eFG%_against_5", "Rolling_eFG%_against_15"
]

PACE_FORM = [
    "Rolling_Pace_5", "Rolling_Pace_15",
    "Rolling_Pace_std_5",
    "Derived_Pace_Shift",
    "Derived_Form_Up",
    "Derived_Form_Up_against",
    "Derived_b2b_x_pace_5",
    "Derived_b2b_x_pace_15"
]

MARKET = [
    "Market_TT",
    "Spread",
    "Is_Home",
    "Is_Favorite"
]

# ===============================
# PCA PIPELINES
# ===============================

def make_pca_block(n):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n, random_state=42))
    ])

pca_attack = make_pca_block(5)
pca_def    = make_pca_block(4)
pca_pace   = make_pca_block(4)

# ===============================
# STORAGE
# ===============================

results  = []
all_bets = []

seasons = sorted(df["season"].unique())

# ===============================
# ROLLING BACKTEST
# ===============================

for i in range(1, len(seasons)):

    train = df[df["season"] < seasons[i]].copy()
    test  = df[df["season"] == seasons[i]].copy()

    # safety
    if len(train) < 500 or len(test) == 0:
        continue

    # -------- PCA FIT --------
    A_tr = pca_attack.fit_transform(train[ATTACK].fillna(0))
    D_tr = pca_def.fit_transform(train[DEFENSE].fillna(0))
    P_tr = pca_pace.fit_transform(train[PACE_FORM].fillna(0))

    X_tr = np.hstack([
        A_tr,
        D_tr,
        P_tr,
        train[MARKET].fillna(0).values
    ])

    y_tr = train["PTS_target"].values

    # -------- MU MODEL --------
    mu_model = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=10.0))
    ])
    mu_model.fit(X_tr, y_tr)

    mu_tr = mu_model.predict(X_tr)
    resid = y_tr - mu_tr

    # -------- SIGMA + NU --------
    sigma = np.clip(np.std(resid, ddof=1), 5.0, 20.0)

    kurt = np.mean((resid / sigma) ** 4)
    nu = max(6, 6 / max(kurt - 3, 0.5))

    # -------- TEST --------
    A_te = pca_attack.transform(test[ATTACK].fillna(0))
    D_te = pca_def.transform(test[DEFENSE].fillna(0))
    P_te = pca_pace.transform(test[PACE_FORM].fillna(0))

    X_te = np.hstack([
        A_te,
        D_te,
        P_te,
        test[MARKET].fillna(0).values
    ])

    mu_raw = mu_model.predict(X_te)

    # ===============================
    # MARKET ANCHORING
    # ===============================
    LAMBDA = 0.6
    market = test["Market_TT"].values
    mu = LAMBDA * mu_raw + (1 - LAMBDA) * market

    # ===============================
    # PROBABILITIES
    # ===============================
    p_over = 1 - t.cdf(market, df=nu, loc=mu, scale=sigma)
    p_under = 1 - p_over

    # ===============================
    # EV CALIBRATION
    # ===============================
    ODDS = 1.91
    EV_SCALE = 0.7

    ev_over  = (p_over  * ODDS - 1) * EV_SCALE
    ev_under = (p_under * ODDS - 1) * EV_SCALE

    # ===============================
    # BET LOGIC
    # ===============================
    pts = test["PTS_target"].values

    side_over = ev_over > ev_under
    best_ev   = np.maximum(ev_over, ev_under)

    bet_side = np.where(
        best_ev < 0.02,
        "NO_BET",
        np.where(side_over, "OVER", "UNDER")
    )

    win = np.where(
        bet_side == "OVER",  pts > market,
        np.where(bet_side == "UNDER", pts < market, np.nan)
    )

    bt = pd.DataFrame({
        "season": seasons[i],
        "bet_side": bet_side,
        "win": win,
        "ev": best_ev
    }, index=test.index)

    all_bets.append(bt)

    placed = bt[bt["bet_side"] != "NO_BET"]

    if len(placed) == 0:
        continue

    results.append({
        "season": seasons[i],
        "bets": len(placed),
        "winrate": np.nanmean(placed["win"]),
        "avg_ev": placed["ev"].mean()
    })

# ===============================
# RESULTS
# ===============================

results_df = pd.DataFrame(results)
print(results_df)

if len(results_df) > 0:
    print("\nAvg bets/season:", results_df["bets"].mean())
    print("Avg winrate:", results_df["winrate"].mean())
    print("Avg EV:", results_df["avg_ev"].mean())
else:
    print("\nNO VALID SEASONS PRODUCED BETS")

bt_all = pd.concat(all_bets)
print("\nEV distribution:")
print(bt_all["ev"].describe())
